# kernelbench: real CUDA validation and measured reports

Choose **Runtime → Change runtime type → GPU**, then run all cells. This notebook has no saved execution outputs: GPU performance is pending until it runs on your hardware. It preserves Colab's CUDA PyTorch installation, checks real numerical GPU tests with **zero accepted skips**, benchmarks FP32 and FP16, and downloads JSON/Markdown reports. Compilation can take several minutes.

The separate distributed experiment validates **CPU Gloo communication and actual DDP/SGD training**. A single CUDA GPU cannot validate multi-GPU NCCL; that result remains explicitly pending. The Triton kernels remain inference-only; DDP uses a separate PyTorch model.


In [ ]:
import importlib.metadata as metadata
import json, os, pathlib, subprocess, sys
import torch

assert torch.cuda.is_available() and torch.version.cuda, "Select a GPU runtime before installation. CPU-only execution is not a GPU result."
original_torch = (str(torch.__version__), torch.version.cuda)
REPOSITORY = "https://github.com/hiroshi-os/kernelbench.git"
REVISION = "main"  # Set a commit SHA for an exactly reproducible rerun.
ROOT = pathlib.Path("/content/kernelbench")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True, timeout=120)
assert subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=ROOT, text=True).strip().removesuffix(".git") == REPOSITORY.removesuffix(".git"), "Existing checkout has another remote"
subprocess.run(["git", "checkout", REVISION], cwd=ROOT, check=True, timeout=30)
os.chdir(ROOT)
# --no-deps prevents pip replacing the runtime's CUDA-enabled torch or its matched Triton.
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."], check=True, timeout=120)
subprocess.run([sys.executable, "-m", "pip", "install", "pytest>=8"], check=True, timeout=120)
installed_torch = subprocess.check_output([sys.executable, "-c", "import torch,json; print(json.dumps([str(torch.__version__),torch.version.cuda]))"], text=True)
assert tuple(json.loads(installed_torch)) == original_torch, "PyTorch changed; restart with a fresh GPU runtime"
print("Preserved CUDA torch:", original_torch)


In [ ]:
import triton  # Colab's CUDA PyTorch normally supplies a compatible Triton.
from kernelbench.ops import gpu_status
from kernelbench.environment import provenance
from kernelbench.launch import bounded_run

assert gpu_status() is None, gpu_status()
# Exercise allocation, device execution and synchronization, not just discovery.
probe = torch.arange(16, device="cuda", dtype=torch.float32).square().sum()
torch.cuda.synchronize()
assert probe.item() == 1240.0
reports = ROOT / "reports"
reports.mkdir(exist_ok=True)
environment = provenance()
environment["triton"] = triton.__version__
environment["preserved_colab_torch"] = list(original_torch)
(reports / "colab-environment.json").write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))
print("If Triton import fails, use a fresh supported CUDA runtime with its matched torch/Triton packages; do not replace torch with a CPU wheel.")


## Mandatory CUDA numerical tests

FP32 and FP16 test both kernels on irregular/masked widths and extreme softmax logits. BF16 is optional because device support varies. `--require-gpu` fails preflight when CUDA/Triton is unavailable, and fails the run if any selected test skips or none are collected. The NCCL test is separate and is not hidden among these kernel checks.


In [ ]:
result = bounded_run([sys.executable, "-m", "pytest", "tests/test_gpu.py", "--require-gpu", "-k", "not bf16", "-q", "-ra", "--junitxml=reports/gpu-tests.xml"], timeout=600)
print(result.stdout)
import xml.etree.ElementTree as ET
suites = ET.parse(reports / "gpu-tests.xml").getroot()
assert sum(int(s.get("tests", 0)) for s in suites.iter("testsuite")) >= 17
assert sum(int(s.get("skipped", 0)) for s in suites.iter("testsuite")) == 0


## FP32 and FP16 measurements

Every backend passes numerical comparison before its CUDA-event timing. Compilation is excluded by warmup. Reports retain every warp configuration, wins and losses, device/driver/software versions, seed, commit and timing distributions. These are kernel operation timings, not end-to-end training speedups.


In [ ]:
from kernelbench.analyze import markdown
from IPython.display import Markdown, display

for dtype in ("float32", "float16"):
    destination = reports / f"gpu-{dtype}.json"
    result = bounded_run([sys.executable, "-m", "kernelbench.benchmark", "--shape", "1x128", "--shape", "128x781", "--shape", "1024x4096", "--dtype", dtype, "--warps", "4", "8", "16", "--warmup", "25", "--samples", "50", "--inner", "20", "--seed", "17", "--output", str(destination)], timeout=1200)
    print(result.stdout)
    table = markdown(json.loads(destination.read_text()))
    destination.with_suffix(".md").write_text(table)
    display(Markdown(table))


## Two-process CPU Gloo: communication and real training

This checks a synchronized all-reduce SUM, then uses DDP on disjoint equal-size shards. Each step compares averaged gradients, global mean loss and updated SGD parameters with a serial full-batch reference. Both processes must pass. Timing is synchronized host wall time and reports the slowest rank per sample. The tiny model measures correctness and harness overhead; it is not a scaling demonstration.


In [ ]:
gloo = reports / "distributed-gloo.json"
result = bounded_run([sys.executable, "-m", "torch.distributed.run", "--standalone", "--nnodes=1", "--nproc-per-node=2", "--max-restarts=0", "-m", "kernelbench.distributed", "--backend", "gloo", "--timeout-seconds", "60", "--output", str(gloo)], timeout=180)
print(result.stdout)
display(Markdown(gloo.with_suffix(".md").read_text()))


## Multi-GPU NCCL: run only on hardware with at least two CUDA GPUs

A typical Colab session exposes one GPU. The next cell writes **pending**, not passed, in that case. It never launches two NCCL workers onto one GPU. To run elsewhere on a Linux machine with two visible GPUs:

```sh
torchrun --standalone --nnodes=1 --nproc-per-node=2 --max-restarts=0 \
  -m kernelbench.distributed --backend nccl --timeout-seconds 60 \
  --output reports/distributed-nccl.json
```

The command hard-fails if fewer than two CUDA GPUs or NCCL support are available. Gloo success cannot validate NCCL, GPU links, or multi-GPU training performance.


In [ ]:
import torch.distributed as dist
nccl_status = {"backend": "nccl", "status": "pending", "visible_cuda_devices": torch.cuda.device_count(), "reason": "Requires at least two visible CUDA GPUs and an NCCL-enabled PyTorch build"}
if torch.cuda.device_count() >= 2 and dist.is_nccl_available():
    nccl = reports / "distributed-nccl.json"
    result = bounded_run([sys.executable, "-m", "torch.distributed.run", "--standalone", "--nnodes=1", "--nproc-per-node=2", "--max-restarts=0", "-m", "kernelbench.distributed", "--backend", "nccl", "--timeout-seconds", "60", "--output", str(nccl)], timeout=180)
    print(result.stdout)
    nccl_status = {"backend": "nccl", "status": "passed", "report": nccl.name}
    display(Markdown(nccl.with_suffix(".md").read_text()))
(reports / "nccl-status.json").write_text(json.dumps(nccl_status, indent=2))
print(json.dumps(nccl_status, indent=2))


## Download this run's evidence

The ZIP includes the raw JSON, measured Markdown comparisons, strict GPU test XML, software/device provenance, Gloo communication/training report, and explicit NCCL status. It excludes the repository's historical CPU reports. Rerun using the recorded commit and comparable hardware before interpreting differences.


In [ ]:
import zipfile
from google.colab import files

archive = ROOT / "kernelbench-colab-reports.zip"
outputs = ["colab-environment.json", "gpu-tests.xml", "gpu-float32.json", "gpu-float32.md", "gpu-float16.json", "gpu-float16.md", "distributed-gloo.json", "distributed-gloo.md", "nccl-status.json"]
if nccl_status["status"] == "passed":
    outputs += ["distributed-nccl.json", "distributed-nccl.md"]
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in outputs:
        bundle.write(reports / filename, arcname=filename)
print("Saved", archive)
files.download(str(archive))


API references: [PyTorch distributed](https://docs.pytorch.org/docs/stable/distributed.html), [DistributedDataParallel](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html), [torchrun](https://docs.pytorch.org/docs/stable/elastic/run.html), [Triton fused softmax](https://triton-lang.org/main/getting-started/tutorials/02-fused-softmax.html).
